# Optional B: How to use a managed search service instead of your own index

**CPU | API key: `OPENAI_API_KEY` (embedding calls; with `MANAGED_SEARCH=on`, uploads, searches and model calls to
OpenAI's file search, OpenAI only) | Self-study**

## Goal

Hand the 120 paragraphs to OpenAI's hosted vector store, search it with the 30 development questions, and weigh what
you gain against what you can no longer see or change. The moment that teaches it: the hosted search matches Lab 05's
pipeline at hit@3, and part of that comes from a ranker you cannot inspect. Switch it off and the right paragraph
drops out of the top three for about 9 of the 30 questions.

## What you will learn

- How to upload documents to a hosted vector store, one file per paragraph, and keep track of every ID you create.
- How to search the store directly and score it with hit@3 and hit@1 against Lab 05's pipeline on the same questions.
- Which settings the service lets you change (chunk size, number of results, the ranker, a score threshold, query
  rewriting) and what stays hidden (the embedding model, the vectors, the ranker).
- How to give the store to a model with the `file_search` tool in the Responses API, and read its query, results and
  citations.
- How to count what the tool's search results cost in input tokens, at 20 results and at 3.
- How to delete every file and store you created, and check from the account listing that none is left.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-24
OPENAI_API_KEY    found in .env


This lab uses `embed_model` and `chat_model` from the lines above, and the switch `MANAGED_SEARCH` from `models.env`.
The other labs use only Chat Completions and Embeddings, which many providers offer. A hosted vector store and the
`file_search` tool are OpenAI's own, so the cells that use them run only when the switch is on. With it empty they do
nothing: read our results in the text and the saved outputs. No other lab depends on this one.

With the switch on, a run stores 120 small files (214 KB, inside the free 1 GB), searches the store about 60
times (not billed) and makes 21 model calls with the `file_search` tool (2.50 US dollars per 1,000 calls): about
0.07 US dollars on `gpt-4o-mini`. The saved outputs come from `gpt-4o-mini`; on another model the counts can move.

In [3]:
# The switch: MANAGED_SEARCH=on in models.env, your .env or an environment variable runs the cells that upload and search.
MANAGED_SEARCH = settings["MANAGED_SEARCH"] == "on"
print("MANAGED_SEARCH:", settings["MANAGED_SEARCH"] or "(empty)", "| cells that upload and search:",
      "run" if MANAGED_SEARCH else "skipped")
s = stamp(settings)
run_line = f"run: {s['chat_model']}, {s['embed_model']}, {s['chat_endpoint']}, {s['run_date']}"

MANAGED_SEARCH: on | cells that upload and search: run


---

## Part A: Measure Lab 05's pipeline on the 30 questions

Part A sets the score the hosted service has to match: Lab 05's pipeline on the same questions.

The service will return whole paragraphs, while Lab 05's store holds chunks. To compare like with like, each hit is
counted by its paragraph, the `source_id` every chunk keeps. Build Lab 05's store from `shared/lc_pipeline.py` and keep
the paragraphs of each question's top three:

In [4]:
import json
from shared import lc_pipeline

questions = json.loads((ROOT / "data" / "development-questions.json").read_text(encoding="utf-8"))
store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
lab05 = [[doc.metadata["source_id"] for doc, _ in lc_pipeline.retrieve(store, q["question"])] for q in questions]
print(len(questions), "questions |", questions[0]["question"], "->", lab05[0])

30 questions | Which NFL team represented the AFC at Super Bowl 50? -> ['S00-P00', 'S00-P01', 'S00-P02']


Two chunks of the same paragraph count once for that paragraph, which is why a list can name a paragraph twice. A
question is a hit at k when its known paragraph is among the first k:

In [5]:
def hit(sources, support, k):
    return any(source in support for source in sources[:k])

def count(ranked, k):
    return sum(hit(sources, q["known_support"], k) for sources, q in zip(ranked, questions))

print(f"Lab 05: hit@3 {count(lab05, 3)} of 30 | hit@1 {count(lab05, 1)} of 30")
print(f"run: {s['embed_model']}, {s['embed_endpoint']}, {s['run_date']}")

Lab 05: hit@3 30 of 30 | hit@1 29 of 30
run: text-embedding-3-small, https://api.openai.com/v1, 2026-10-24


Lab 05's pipeline finds the right paragraph in the top three for 30 of 30 questions and first for 29.
Everything behind those numbers is yours to read: the chunks, the embedding model, every vector and the cosine that
ranks them.

### Checkpoint A

| You should see | Our run |
|---|---|
| The first question | `['S00-P00', 'S00-P01', 'S00-P02']` |
| Lab 05 | hit@3 30, hit@1 29 of 30 |

---

## Part B: Upload the paragraphs to a hosted vector store

Part B hands the 120 paragraphs to OpenAI, which chunks, embeds and indexes them for you: the steps of Labs 01 and 03,
done by the service.

A **vector store** is OpenAI's hosted index ([retrieval guide](https://developers.openai.com/api/docs/guides/retrieval)).
It splits each file into chunks of up to 800 tokens, and a search result names its file. So upload one `.txt` file
per paragraph, named after its source ID, and every result names its paragraph. One file holding all 120 paragraphs
would be cut into chunks that mix several paragraphs.

Make the files in memory first. Nothing is sent yet:

In [6]:
passages = json.loads((ROOT / "data" / "squad-passages.json").read_text(encoding="utf-8"))
files = [(p["source_id"] + ".txt", p["text"].encode("utf-8")) for p in passages]
print(len(files), "files | first:", files[0][0], files[0][1][:60])

120 files | first: S00-P00.txt b'Super Bowl 50 was an American football game to determine the'


⚠️ From here on, the cells that start with `if MANAGED_SEARCH:` use OpenAI's hosted search, and what they upload
stays in your account until you delete it. Every ID the notebook creates goes into `created`, and the last section deletes exactly those, never anything
else in your account. If a cell fails halfway, run the last section before you start again.

Upload each file with the Files API (`purpose="assistants"` is the purpose vector stores read):

In [7]:
from shared.settings import chat_client

created = {"files": [], "stores": []}
if MANAGED_SEARCH:
    client = chat_client(settings)
    for name, content in files:
        created["files"].append(client.files.create(file=(name, content), purpose="assistants").id)
    print("uploaded:", len(created["files"]), "files")

uploaded: 120 files


Now create a store and add the files in one batch. `create_and_poll` waits until every file is chunked and
embedded. `expires_after` is a safety net: a store nobody searches for a day deletes itself. Its files do not, as the
last section shows.

In [8]:
if MANAGED_SEARCH:
    vector_store = client.vector_stores.create(name="rag_for_nlp Optional B",
                                               expires_after={"anchor": "last_active_at", "days": 1})
    created["stores"].append(vector_store.id)
    batch = client.vector_stores.file_batches.create_and_poll(vector_store.id, file_ids=created["files"])
    print(batch.status, "|", batch.file_counts.completed, "of", batch.file_counts.total, "files indexed")

in_progress | 120 of 120 files indexed


What did the service do with them? Ask the store about one file and about itself:

In [9]:
if MANAGED_SEARCH:
    one = client.vector_stores.files.retrieve(created["files"][0], vector_store_id=vector_store.id)
    print("chunking:", one.chunking_strategy.static)
    print("store size:", client.vector_stores.retrieve(vector_store.id).usage_bytes, "bytes")

chunking: StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800)


store size: 214148 bytes


The store reports its chunking (800 tokens, 400 overlap; each paragraph fits in one chunk) and its size,
214,148 bytes. Nothing tells you which embedding model it used, and no call returns a vector.

### Checkpoint B

| You should see | Our run |
|---|---|
| Files | 120, the first `S00-P00.txt` |
| Store | `completed`, 120 of 120 files indexed |
| Inspect | `chunk_overlap_tokens=400`, `max_chunk_size_tokens=800`; 214,148 bytes |

---

## Part C: Search the store and compare it with Lab 05

Part C asks the store the 30 questions and compares its answers with Lab 05's, question by question.

`vector_stores.search` takes a question and returns the closest chunks, each with its file name, a score and the text.
Try the first question:

In [10]:
if MANAGED_SEARCH:
    results = client.vector_stores.search(vector_store.id, query=questions[0]["question"])
    print(questions[0]["question"], "|", len(results.data), "results")
    for r in results.data[:4]:
        print(f"{r.score:.4f}  {r.filename}  {r.content[0].text[:60]}")

Which NFL team represented the AFC at Super Bowl 50? | 10 results
0.9599  S00-P00.txt  Super Bowl 50 was an American football game to determine the
0.8162  S00-P02.txt  The Broncos took an early lead in Super Bowl 50 and never tr
0.8078  S00-P01.txt  The Panthers finished the regular season with a 15–1 record,
0.0384  S24-P01.txt  ABC originally launched on October 12, 1943 as a radio netwo


Ten results by default, best first, each a whole paragraph. The three Super Bowl paragraphs score 0.96, 0.82 and 0.81, and the
fourth falls to 0.04. A cosine between two `text-embedding-3-small` vectors does not fall like that, so this
score is something else; we come back to it in Part D.

Now the 30 questions. The file name gives the paragraph back:

In [11]:
def managed_sources(**options):
    ranked = []
    for q in questions:
        results = client.vector_stores.search(vector_store.id, query=q["question"], **options)
        ranked.append([r.filename.removesuffix(".txt") for r in results.data])
    return ranked

if MANAGED_SEARCH:
    managed = managed_sources()
    for k in (3, 1):
        print(f"hit@{k}: Lab 05 {count(lab05, k)} | managed {count(managed, k)} of 30")
    print(run_line)

hit@3: Lab 05 30 | managed 30 of 30
hit@1: Lab 05 29 | managed 28 of 30
run: gpt-4o-mini, text-embedding-3-small, https://api.openai.com/v1, 2026-10-24


Which questions does one of them put first and the other not?

In [12]:
if MANAGED_SEARCH:
    for s05, sm, q in zip(lab05, managed, questions):
        if hit(s05, q["known_support"], 1) != hit(sm, q["known_support"], 1):
            print(f"Lab 05 {s05} | managed {sm[:3]} | known {q['known_support']} | {q['question'].strip()}")

Lab 05 ['S00-P00', 'S00-P02', 'S00-P01'] | managed ['S00-P02', 'S00-P01', 'S00-P00'] | known ['S00-P02'] | Who was the Super Bowl 50 MVP?
Lab 05 ['S04-P00', 'S04-P01', 'S04-P02'] | managed ['S04-P01', 'S04-P02', 'S04-P00'] | known ['S04-P00'] | By what main attribute are computational problems classified utilizing computational complexity theory?
Lab 05 ['S04-P02', 'S04-P01', 'S04-P02'] | managed ['S04-P01', 'S04-P02', 'S04-P00'] | known ['S04-P02'] | What field of computer science analyzes the resource requirements of a specific algorithm isolated unto itself within a given problem?


Both find the right paragraph in the top three for all 30 questions. At hit@1 the store gains one question and loses two: it puts the MVP paragraph first, where Lab 05 put the game summary, and for two complexity theory questions it puts a neighbouring paragraph first, where Lab 05 had the right one. On these questions the hosted store does as well as your own pipeline, and you wrote no chunking, embedding or index code.

### Checkpoint C

| You should see | Our run |
|---|---|
| One question | 10 results; `S00-P00.txt` first; three scores near 0.81 to 0.96, then 0.04 |
| 30 questions | Lab 05 hit@3 30, hit@1 29; managed hit@3 30, hit@1 28 |

---

## Part D: See what the ranker does, and what you can change

Part D switches off the one stage the service lets you turn off, its ranker, to see how much of Part C's score it
carries.

A search takes `ranking_options` with a `ranker`: `auto`, the default, or `none`. The API reference says only that
`none` "can help reduce latency". Before you run it: how many of the 30 questions will keep their paragraph in the
top three?

In [13]:
if MANAGED_SEARCH:
    unranked = managed_sources(ranking_options={"ranker": "none"})
    for k in (3, 1):
        print(f"hit@{k}: ranker auto {count(managed, k)} | ranker none {count(unranked, k)} of 30")
    print(run_line)

hit@3: ranker auto 30 | ranker none 22 of 30
hit@1: ranker auto 28 | ranker none 13 of 30
run: gpt-4o-mini, text-embedding-3-small, https://api.openai.com/v1, 2026-10-24


Without the ranker, hit@3 falls from 30 to 22 and hit@1 from 28 to 13 (hit@3 was 20 to 22 in our seven runs). Look at the scores without the ranker:

In [14]:
if MANAGED_SEARCH:
    results = client.vector_stores.search(vector_store.id, query=questions[0]["question"],
                                          ranking_options={"ranker": "none"})
    for r in results.data[:4]:
        print(f"{r.score:.4f}  {r.filename}")
    print(f"1/61 + 1/62 = {1/61 + 1/62:.4f}")

0.0325  S00-P01.txt
0.0315  S00-P02.txt
0.0311  S00-P00.txt
0.0241  S09-P02.txt
1/61 + 1/62 = 0.0325


The top three now score 0.0311 to 0.0325, and the answer `S00-P00` comes third. The first is exactly 1/61 + 1/62 = 0.0325. These scores match reciprocal rank fusion with c = 60 from Lab 08: each is close to a sum of 1/(60
+ rank) over two ranked lists, and first in one list with second in the other gives 1/61 + 1/62. So the search
without the ranker looks like an embedding list and a keyword list fused by rank (the retrieval guide documents this
hybrid search). The ranker then re-scores the fused list, and Part C's 0.96 and 0.04 are probably its own scores. Lab 08's reranker moved two questions at hit@1 and none at hit@3. This one lifts hit@3 from
22 to 30, and you cannot read its prompt, its model or why it chose.

| | Hosted vector store | Lab 05's pipeline |
|---|---|---|
| Chunking | size and overlap, set when you add files | your own code (Lab 03) |
| Embedding model, vectors | hidden | `EMBED_MODEL`; every vector |
| Keyword search and fusion | always on; weights in the `file_search` tool | not used (Lab 08) |
| Reranker | `auto` or `none`; a `score_threshold` | none, or your own (Lab 08) |
| Results | `max_num_results`: 10 here, 20 in the tool | `k=3` |
| Query | `rewrite_query`, off by default | sent as written |
| Billed | storage beyond 1 GB, tool calls | embedding tokens |

### Checkpoint D

| You should see | Our run |
|---|---|
| Ranker `none` | hit@3 22, hit@1 13 of 30, against 30 and 28 with `auto` |
| Scores without the ranker | about 0.03 at the top; `1/61 + 1/62 = 0.0325` |

---

## Part E: Let a model search with the `file_search` tool

Part E gives the store to a model as a hosted tool, the way most applications use it, and counts what the search
results cost in tokens.

`file_search` is a tool OpenAI runs for the model: the model writes a query, the service searches the store, adds the
results to the model's input, and the model answers with citations. It exists only in the Responses API, the one place
these labs call `client.responses.create` ([file search guide](https://developers.openai.com/api/docs/guides/tools-file-search)).
The options come from the same settings as every chat call; a reasoning model's effort goes in another shape here:

In [15]:
from shared.settings import chat_options

options = chat_options(settings=settings)
if "reasoning_effort" in options:
    options["reasoning"] = {"effort": options.pop("reasoning_effort")}
print(options)

{'model': 'gpt-4o-mini', 'temperature': 0}


Ask the first question. `tool_choice` makes the model search every time, so each question costs exactly one tool call,
and `include` asks for the results the model read:

In [16]:
if MANAGED_SEARCH:
    tool = {"type": "file_search", "vector_store_ids": [vector_store.id]}
    response = client.responses.create(input=questions[0]["question"], tools=[tool], tool_choice={"type": "file_search"},
                                       include=["file_search_call.results"], **options)
    call = next(item for item in response.output if item.type == "file_search_call")
    print("output items:", [item.type for item in response.output])
    print("query the model wrote:", call.queries, "| results it read:", len(call.results))

output items: ['file_search_call', 'message']
query the model wrote: ['Which NFL team represented the AFC at Super Bowl 50?'] | results it read: 20


The model wrote its own query and read 20 results, twice the store's default. Its answer, the files it
cites and the input tokens it was billed:

In [17]:
if MANAGED_SEARCH:
    message = next(item for item in response.output if item.type == "message")
    print("answer:", response.output_text)
    print("cited:", [a.filename for a in message.content[0].annotations])
    print("input tokens:", response.usage.input_tokens)

answer: The NFL team that represented the AFC at Super Bowl 50 was the Denver Broncos. They defeated the Carolina Panthers with a score of 24–10.
cited: ['S00-P00.txt']
input tokens: 5515


*Denver Broncos*, citing `S00-P00.txt`, for 5,515 input tokens. The question is ten words; the rest of the input is the 20 paragraphs the tool added. `max_num_results` in
the tool sets how many it adds. Compare 20 with 3 on every third development question (10 questions, so 20 tool
calls). `found()` from Lab 06 checks whether a whole accepted answer appears in the reply:

In [18]:
from shared.evals import found

def ask(question, tool):
    return client.responses.create(input=question, tools=[tool], tool_choice={"type": "file_search"}, **options)

if MANAGED_SEARCH:
    sample = questions[::3]
    tools = {"20 results (default)": tool, "3 results": {**tool, "max_num_results": 3}}
    runs = {}
    for name, run_tool in tools.items():
        replies = runs[name] = [ask(q["question"], run_tool) for q in sample]
        tokens = sum(r.usage.input_tokens for r in replies)
        right = sum(found(r.output_text, [a["text"] for a in q["answers"]]) for r, q in zip(replies, sample))
        dollars = tokens * float(settings["CHAT_PRICE_INPUT"]) / 1e6
        print(f"{name:21} | input tokens {tokens:7,} | {dollars:.4f} USD | answer found {right} of {len(sample)}")
    print(run_line)

20 results (default)  | input tokens  51,532 | 0.0077 USD | answer found 8 of 10


3 results             | input tokens  18,422 | 0.0028 USD | answer found 8 of 10
run: gpt-4o-mini, text-embedding-3-small, https://api.openai.com/v1, 2026-10-24


Print the replies the check did not pass, so you can see whether the answer was wrong or only worded differently:

In [19]:
if MANAGED_SEARCH:
    for name, replies in runs.items():
        for r, q in zip(replies, sample):
            if not found(r.output_text, [a["text"] for a in q["answers"]]):
                print(f"{name} | {q['question'].strip()} | accepted {q['answers'][0]['text']!r}\n   {r.output_text[:160]}")

20 results (default) | What is a popular strolling destination for the Varsovians? | accepted 'Krasiński Palace Garden'
   A popular strolling destination for Varsovians is the **Saxon Garden**, which features over 100 species of trees and provides a relaxing atmosphere with benches
20 results (default) | What method is used to intuitively assess or quantify the amount of resources required to solve a computational problem? | accepted 'mathematical models of computation'
   The method used to intuitively assess or quantify the amount of resources required to solve a computational problem is primarily through **computational complex
3 results | What is a popular strolling destination for the Varsovians? | accepted 'Krasiński Palace Garden'
   A popular strolling destination for Varsovians is the **Saxon Garden**, which features over 100 species of trees and provides a relaxing atmosphere with benches
3 results | What method is used to intuitively assess or quantify the amount of resourc

Three results cost about a third of the input tokens of twenty (18,422 against 51,532) and found the accepted answer as often, 8 of 10. The same two replies failed both times, and both are wrong, not reworded: the Saxon Garden for the Varsovians' strolling destination (the paragraph names the Krasiński Palace Garden), and *computational complexity theory* for the method. More results did not help, and the tool fee for 10 calls (0.025 US dollars) is larger than either token bill. Twenty is a default, not a choice made for your documents.

### Checkpoint E

| You should see | Our run |
|---|---|
| One question | `['file_search_call', 'message']`; the model's own query; 20 results; *Denver Broncos*, citing `S00-P00.txt`; 5,515 input tokens |
| 20 against 3 results | 51,532 against 18,422 input tokens; answer found 8 and 8 of 10 |

---

## What you can do now

- Upload documents to a hosted vector store, one file per paragraph, keeping every ID you create.
- Search the store directly and compare it with your own pipeline at hit@3 and hit@1 on the same questions.
- Say which settings the service exposes (chunking, number of results, ranker, score threshold, query rewriting) and
  what it keeps hidden (the embedding model, the vectors, the ranker's model).
- Call a model with the `file_search` tool and read its query, results and citations.
- Count the input tokens the tool's results add, and cut them with `max_num_results`.
- Delete every file and store you created, and check the account listing (the last section, below the challenge).

## Challenge

Use only what this lab showed. Task 1 is the challenge; task 2 is
optional. Both need the store, so do them before the last section deletes it. For each task, keep the printed results
and your answer to **Explain**.

1. **Let the service rewrite the questions.** `vector_stores.search` takes `rewrite_query=True`: the service rewrites
   the question before it searches, and returns the query it used in `results.search_query`.

   - Complete the loop under **Task 1**. The cell prints hit@3, hit@1 and every query that was rewritten.

   **Check:** hit@3 and hit@1 against Part C's. Our worked solution got hit@3 30 and hit@1 28, the same as without rewriting, while 21 of the 30 queries were rewritten, mostly into keyword phrases (*Super Bowl 50 MVP*), and one corrected *Maria Curie* to *Marie Curie*.

   **Explain:** Lab 09 rewrites follow-up questions by hand. What does a rewrite you cannot switch per question cost
   you when you debug a miss?

2. **Optional: keep only results above a score.** Under **Task 2**, search the 30 questions with
   `ranking_options={"ranker": "auto", "score_threshold": 0.5}`.

   **Check:** the average number of results per question and hit@3. Our worked solution kept 1.5 results per question on average, and hit@3 stayed at 30 of 30.

   **Explain:** what would a threshold do to Part E's input tokens, and what could it cost when the answer scores low?

**Task 1.** Search with `rewrite_query=True`:

In [20]:
if MANAGED_SEARCH:
    rewritten, queries = [], []
    for q in questions:
        # TODO: search the store for q["question"] with rewrite_query=True, append the file names without ".txt"
        #       to rewritten (as managed_sources() does) and results.search_query to queries
        pass
    if not rewritten:
        print("⚠️ Complete the loop, then run the cell again.")
    else:
        print(f"rewritten: hit@3 {count(rewritten, 3)} | hit@1 {count(rewritten, 1)} of 30")
        for q, query in zip(questions, queries):
            if query != [q["question"]]:
                print(f"{q['question']!r} -> {query}")

⚠️ Complete the loop, then run the cell again.


**Task 2 (optional).** Keep only results scoring 0.5 or more:

In [21]:
if MANAGED_SEARCH:
    kept = []
    # TODO: call managed_sources() with ranking_options={"ranker": "auto", "score_threshold": 0.5} and store it in kept
    if not kept:
        print("⚠️ Search with the threshold, then run the cell again.")
    else:
        print(f"results per question: {sum(len(s) for s in kept) / len(kept):.1f} | hit@3 {count(kept, 3)} of 30")

⚠️ Search with the threshold, then run the cell again.


---

## Delete what this notebook created

Run these two cells last, also if you skipped the challenge. Storage under 1 GB is free, but an uploaded file stays in
your account until you delete it.

First delete the store, then look for your files in the account:

In [22]:
if MANAGED_SEARCH:
    print("files in the account:", len(list(client.files.list())),
          "| stores in the account:", len(list(client.vector_stores.list())))
    for store_id in created["stores"]:
        print("store deleted:", client.vector_stores.delete(store_id).deleted)
    ours = [f.id for f in client.files.list() if f.id in created["files"]]
    print("our files still in the account:", len(ours), "of", len(created["files"]))

files in the account: 128 | stores in the account: 3


store deleted: True


our files still in the account: 120 of 120


The store is gone, and all 120 of its files are still in the account: deleting a store removes its index, not the files you uploaded. Delete each file by its ID, then list the account's files and stores and look for any ID this notebook
created:

In [23]:
if MANAGED_SEARCH:
    for file_id in created["files"]:
        client.files.delete(file_id)
    account_files = [f.id for f in client.files.list()]
    account_stores = [store.id for store in client.vector_stores.list()]
    print("files in the account:", len(account_files), "| stores in the account:", len(account_stores))
    files_left = [file_id for file_id in account_files if file_id in created["files"]]
    stores_left = [store_id for store_id in account_stores if store_id in created["stores"]]
    print(f"left in the account: {len(files_left)} of our {len(created['files'])} files, "
          f"{len(stores_left)} of our {len(created['stores'])} stores")

files in the account: 8 | stores in the account: 2
left in the account: 0 of our 120 files, 0 of our 1 stores


The account went from 128 files and 3 stores to 8 files and 2 stores: down by exactly the notebook's 120 files and one store. The listing found none of them left, and nothing else was touched.

Build or buy? The hosted store saved you the index code and matched your pipeline at hit@3, with a ranker you can
switch off but not read. Your own pipeline is more code and shows every step, which Labs 04, 06 and 08 needed to find
and fix a miss. And the hosted store ties you to OpenAI.

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| The cells after Part A print nothing | `MANAGED_SEARCH` is empty | Set `MANAGED_SEARCH=on` in your `.env` (or `models.env`) and run Config and the switch cell again |
| `NameError: name 'client' is not defined` in a later cell | The switch was turned on after Part B ran | Run Config and the switch cell, then the notebook again from Part B |
| A search returns results from files you thought you removed | Removing files from a store is eventually consistent | Wait a short time and search again; this lab deletes the whole store |
| Your account still holds files named `S00-P00.txt` and so on after a failed run | A cell failed before the last section | Run the last section; if the kernel restarted, `created` is empty, so delete them on the Storage page of the OpenAI dashboard |

## Resources

- [OpenAI: retrieval guide](https://developers.openai.com/api/docs/guides/retrieval): vector stores, search, ranking
  options, query rewriting, chunking and storage pricing
- [OpenAI: file search tool](https://developers.openai.com/api/docs/guides/tools-file-search): the `file_search` tool
  in the Responses API, `max_num_results` and `include`
- [OpenAI: pricing](https://developers.openai.com/api/docs/pricing): file search storage and tool call prices
- [OpenAI API reference: vector stores](https://developers.openai.com/api/reference/resources/vector_stores): every
  method, including `delete` for stores and for files

Next: [Optional A: How to build the same pipeline with LlamaIndex](../Optional_LlamaIndex_Pipeline/README.md). Back to
[Lab 05](../05_Rebuild_With_LangChain/README.md), the pipeline this lab compares with, or to [the lab list](../README.md).